# Collections

## Inventaire

On souhaite calculer le stock disponible d'une boutique à partir d'une liste de transactions de la forme `(category, item, quantity)` où `quantity` est positif ou négatif (achat ou vente).

Utilisez une ou des structures de données efficaces (du module [`collections`](https://docs.python.org/fr/3/library/collections.html) par exemple) pour calculer et stocker cet inventaire.

In [ ]:
operations = [
  ("nourriture", "haricots", 5),
  ("nourriture", "maïs", 10),
  ("vêtements", "tshirt", 20),
  ("vêtements", "tshirt", -1),
  ("nourriture", "maïs", -2),
  ("nourriture", "haricots", -1),
  ("vêtements", "tshirt", -1),
  ("nourriture", "maïs", -3),
  ("vêtements", "tshirt", -1),
]


# Votre code ici

### Solution

In [ ]:
from collections import Counter, defaultdict


inventory = defaultdict(Counter)
for category, item, quantity in operations:
  inventory[category][item] += quantity


print(inventory)
print({k: dict(v) for k, v in inventory.items()})

## `defaultdict` imbriqués

Créez une structure de données qui fera correspondre des auteurs aux années où ils ont écrit leurs livres, ces années à ces livres, puis ces livres à leur texte representé comme une liste de mots. Pour cela, utilisez deux `defaultdict` imbriqués, et utilisez simplement la méthode `split` pour séparer le texte en mots, même si c'est imparfait.

In [ ]:
books = [
    ("V. Hugo", 1862, "Les Misérables", "En  1815,  M.  Charles-François-Bienvenu  Myriel était  évêque  de  Digne."),
    ("V. Hugo", 1831, "Notre-Dame de Paris", "Il y a aujourd’hui trois cent quarante-huit ans six mois et dix-neuf jours…"),
    ("L. Tolstoï", 1865, "Guerre et Paix", "« Eh bien, prince, que vous disais-je ? Gênes et Lucques sont devenues…")
]

# Votre code ici

### Solution

In [ ]:
from collections import defaultdict
from pprint import pprint


books_dict = defaultdict(lambda: defaultdict(dict))
for author, year, title, text in books:
  books_dict[author][year][title] = text.split()
pprint(books_dict)

## Fusion de configuration

Étant donné deux configurations, l'une globale, l'autre spécifique, créez un dictionnaire qui contient les valeurs des deux fusionnées, en donnant la priorité à la spécifique quand il y a conflit.

In [ ]:
global_configuration = dict(
    font_size=10,
    left_margin=1,
    right_margin=2,
)


specific_configuration = dict(
    font_size=11,
    top_margin=3
)


# Votre code ici

### Solution

In [ ]:
import collections


merged_configuration = collections.ChainMap(specific_configuration,
                                            global_configuration)
for k, v in merged_configuration.items():
  print(k, v)

# Découverte de `pandas`

Pour toute la durée de cette découverte, gardez la [feuille d'aide `pandas`](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) à disposition.

In [ ]:
import re

import pandas
import seaborn

## Récupération des données
Exécutez la cellule suivante pour récupérer des données textuelles.

C'est une collection de trames de scénarios de différents films ainsi que des méta-données telles que l'année de sortie, le titre ou le genre.

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-wikipedia-movie-plots.git
movie_plot_path = "dataset-wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Chargement dans un dataframe pandas

Chargez dans un dataframe panda les données contenues dans le fichier `.csv` avec la fonction [`pandas.read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html).
Vérifiez avec l'attribut `columns` des dataframes que les colonnes sont cohérentes avec le `.csv`
Trouvez le nombre de films dans la base à l'aide de l'attribut `shape` des dataframes.

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pandas.read_csv(movie_plot_path)
print(f"Colonnes : {', '.join(df.columns)}")
print(f"Forme de la df : {df.shape}")

## Démonstration de sélection avancée

Nous allons maintenant voir comment conserver seulement les films qui ont un genre connnu (et donc supprimer les films dont le genre est `unknown`). On va ensuite conserver seulement les films qui ont pour genre l'un des 5 genres les plus représentés.

Fonctions utilisées :

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)

In [ ]:
counts = df["Genre"].value_counts().drop("unknown")
genres = counts.index.values[:5]
print(f"Genres conservés : {', '.join(genres)}")
print(f"Nombre de films conservés : {counts[genres].sum()}")

In [ ]:
clean_df = df[df["Genre"].isin(genres)]
seaborn.countplot(x=clean_df["Genre"])

## Démonstration de transformation de données

Nous allons remplacer les nombres dans les scénarios des films par un mot spécial (c'est une technique souvent utilisée en apprentissage automatique par exemple, pour généraliser sur les nombres).

In [ ]:
regex = re.compile(r"[0-9]+")


def preprocess(text: str) -> str:
  return regex.sub("aanumber", text)


clean_df["Cleaned Plot"] = clean_df["Plot"].map(preprocess)
clean_df.iloc[:100, :]